# Module `algorithms/simulated_annealing.py`

Le recuit simule explore le voisinage en acceptant parfois des degradations, avec une probabilite qui decroit avec la temperature. C'est un *single-solution metaheuristic* : un unique point courant evolue iteration par iteration.

Critere de Metropolis :

```
P(accepter) = 1                     si delta < 0
            = exp(-delta / T)       sinon
```

Refroidissement geometrique : `T <- cooling_rate * T` a chaque iteration.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

import math
import random
from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import simulated_annealing
from cesipath.solver_input import build_static_solver_input


## 1. Instance et appel standard


In [2]:
cfg = GraphGenerationConfig(node_count=18, seed=11)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)

sol = simulated_annealing(
    si,
    initial_temperature=100.0,
    cooling_rate=0.97,
    max_iterations=1500,
    seed=0,
)
print(f"cout = {sol.total_cost:.2f}  ; routes = {sol.route_count}")


cout = 599.90  ; routes = 1


## 2. Lecture des parametres

| Parametre | Role |
|---|---|
| `initial_temperature` | Haute -> accepte quasiment tout. Doit permettre d'accepter la plupart des degradations au debut. |
| `final_temperature` | Seuil d'arret : quand `T <= final_temperature`, on stoppe. |
| `cooling_rate` | Dans (0, 1). Proche de 1 (ex. 0.995) -> refroidissement lent -> plus de diversification. Proche de 0.9 -> descente rapide. |
| `max_iterations` | Arret fort en nombre d'iterations. Prend le dessus si la temperature descend lentement. |
| `initial_rcl_alpha` | Construction initiale via GRASP : 0 = plus proche voisin pur, >0 = randomise. |
| `final_local_search` | Descente deterministe sur la meilleure solution rencontree en fin de parcours (recommande). |


## 3. Effet du refroidissement


In [3]:
for cr in (0.90, 0.97, 0.995):
    sol = simulated_annealing(
        si,
        initial_temperature=100.0,
        cooling_rate=cr,
        max_iterations=5000,
        seed=1,
    )
    print(f"cooling_rate = {cr:>5} -> cout = {sol.total_cost:.2f}")


cooling_rate =   0.9 -> cout = 599.90
cooling_rate =  0.97 -> cout = 599.90
cooling_rate = 0.995 -> cout = 619.99


## 4. Probabilite d'acceptation en fonction de T

Pour une meme degradation `delta`, la probabilite d'acceptation chute vite quand `T` descend. C'est la propriete fondamentale qui fait basculer l'algo de l'exploration (haute temperature) a l'intensification (basse temperature).


In [4]:
delta = 5.0
for T in (100.0, 30.0, 10.0, 3.0, 1.0, 0.3):
    p = math.exp(-delta / T)
    print(f"T={T:>6.2f}   P(accepter delta=+{delta}) = {p:.4f}")


T=100.00   P(accepter delta=+5.0) = 0.9512
T= 30.00   P(accepter delta=+5.0) = 0.8465
T= 10.00   P(accepter delta=+5.0) = 0.6065
T=  3.00   P(accepter delta=+5.0) = 0.1889
T=  1.00   P(accepter delta=+5.0) = 0.0067
T=  0.30   P(accepter delta=+5.0) = 0.0000


## 5. Impact de `final_local_search`

Le recuit navigue avec des mouvements aleatoires : meme en fin de parcours il peut rester dans un voisinage sous-optimal deterministe. La passe finale `local_search` polit la meilleure solution rencontree.


In [5]:
a = simulated_annealing(si, max_iterations=2000, cooling_rate=0.97, final_local_search=True,  seed=5)
b = simulated_annealing(si, max_iterations=2000, cooling_rate=0.97, final_local_search=False, seed=5)
print(f"avec passe finale : {a.total_cost:.2f}")
print(f"sans passe finale : {b.total_cost:.2f}")


avec passe finale : 599.90
sans passe finale : 670.34


## 6. Variance sur plusieurs graines

Le recuit est stochastique : une seule execution ne represente pas sa performance moyenne. En benchmark, on lance plusieurs graines par instance.


In [6]:
costs = []
for seed in range(8):
    sol = simulated_annealing(si, max_iterations=1500, cooling_rate=0.97, seed=seed)
    costs.append(sol.total_cost)
print(f"n={len(costs)}  min={min(costs):.2f}  max={max(costs):.2f}  moy={sum(costs)/len(costs):.2f}")


n=8  min=599.90  max=628.37  moy=603.46


## 7. Reproductibilite


In [7]:
a = simulated_annealing(si, max_iterations=500, seed=9)
b = simulated_annealing(si, max_iterations=500, seed=9)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True


## 8. En resume

- SA excelle pour sortir des minima locaux grace au critere de Metropolis.
- Necessite un tuning du couple `(initial_temperature, cooling_rate)` : trop froid -> descente gloutonne ; trop chaud -> marche aleatoire.
- Toujours activer `final_local_search` sauf etude specifique.
